# M3 — corrección v6: diagnóstico, no evaluación final
Usa una sesión nueva de Colab con GPU T4. Sube únicamente `contractrisk_m3_correccion_v6_codigo.zip`. Se reutilizan los contextos de desarrollo v5 para aislar cambios de generación; el agente vuelve a decidir y ejecutar la herramienta. Se guardan 32 respuestas y 48 juicios sintéticos. No se ejecuta el gold ni RAGAS. La latencia no es comparable con recuperación real. Conserva los resultados aunque fallen; el éxito técnico no certifica calidad.

In [1]:
from google.colab import files
from pathlib import Path
import tempfile, zipfile, os, sys, subprocess, shutil
uploaded = files.upload()
if len(uploaded) != 1: raise ValueError('Sube un solo ZIP de código v6')
ROOT = Path(tempfile.mkdtemp(prefix='contractrisk_m3_v6_', dir='/content'))
with zipfile.ZipFile(next(iter(uploaded))) as archive:
    for name in archive.namelist():
        if not (ROOT / name).resolve().is_relative_to(ROOT): raise ValueError('Ruta ZIP insegura')
    archive.extractall(ROOT)
os.chdir(ROOT)
assert (ROOT/'m3/run_correction_v6.py').exists()
print(ROOT)


Saving contractrisk_m3_correccion_v6_codigo.zip to contractrisk_m3_correccion_v6_codigo.zip
/content/contractrisk_m3_v6_9ny09o0m


In [2]:
def logged(command, destination):
    with open(destination, 'w', encoding='utf-8') as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace')
        for line in process.stdout:
            print(line, end=''); log.write(line); log.flush()
        code = process.wait()
    if code: raise subprocess.CalledProcessError(code, command)
logged([sys.executable, '-m', 'pip', 'install', '-r', 'm3/requirements-development-v2.txt'], ROOT/'installation.log')
import torch
assert torch.cuda.is_available(), 'Activa GPU T4'
print('GPU:', torch.cuda.get_device_name(0))
logged([sys.executable, '-m', 'unittest', 'discover', '-s', 'm3/tests', '-v'], ROOT/'tests.log')


  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 30.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 30.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 44.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 78.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 80.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 103.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 131.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 63.7 MB/s

In [3]:
# No repetir con la misma carpeta: conserva cualquier evidencia anterior.
logged([sys.executable, '-u', 'm3/run_correction_v6.py', '--output', 'v6_resultados'], ROOT/'execution.log')


`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1793293064.505784    2980 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793293068.723332    2980 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.

Fetching 2 files: 100%|██████████| 2/2 [00:55<00:00, 27.51s/it]

Loading checkpoint shards: 100%|██████████| 2/2 [00:25<00:00, 12.61s/it]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
control_no_rag_v6_replay 0
control_no_rag_v6_replay 1
control_no_rag_v6_replay 2
control_no_rag_v6_replay 3
control_no_rag_v6_replay 4
control_no_rag_v6_replay 5
control_no_rag_v6_replay 6
control_no_rag_v6_replay 7
dense_v6_replay 

In [4]:
# Ejecuta también esta celda si falla la anterior.
OUT = ROOT/'v6_resultados'
OUT.mkdir(exist_ok=True)
for filename in ['installation.log','tests.log','execution.log']:
    if (ROOT/filename).exists(): shutil.copy2(ROOT/filename, OUT/filename)
import json
if (OUT/'metadata.json').exists():
    print((OUT/'metadata.json').read_text(encoding='utf-8'))
if (OUT/'calibration_summary.json').exists():
    summary=json.loads((OUT/'calibration_summary.json').read_text(encoding='utf-8'))
    print(json.dumps({k:{a:b for a,b in v.items() if a!='pairs'} for k,v in summary.items()}, indent=2))
path=shutil.make_archive(str(ROOT/'contractrisk_m3_correccion_v6_resultados'), 'zip', OUT)
files.download(path)
print('Devuelve este ZIP y descarga el notebook con sus salidas.')


{
  "status": "completed",
  "purpose": "development_context_replay_and_judge_calibration_NOT_final",
  "started_utc": "2026-10-29T16:57:31.943622+00:00",
  "seed": 42,
  "python": "3.13.16",
  "quality_status": "requires_semantic_review",
  "rebuilds_retrieval": false,
  "gpu": "Tesla T4",
  "model": "Qwen/Qwen2.5-3B-Instruct",
  "revision": "aa8e72537993ba99e69dfaafa59ed015b17504d1",
  "judge_same_model": true,
  "replay_zip_sha256": "a12b297d7f8854c6ef24b8bef66b910cf38a6b5d4545a0a53d3b20965bf772c0",
  "source_hashes": {
    "freeze_gold.py": "f78b353ebcc66dc1f9a7617df3e9ce0ef6b3c69e025c93870dce4e383e815698",
    "systems_v5.py": "b160cb8b35e3e6e7bdba2ffe140923d7cd612ad8935e43f9fe49ab9ce797fafd",
    "prepare_eval_draft.py": "dbdbb424f2fb2979994971b1612b627047ee7ae91a266fc7eac9dfa4ff62482d",
    "run_agent_development.py": "c442315fa07d387ea2135966b01205e3339a9c4b94701bbb942b6f8e68c1461f",
    "ingest.py": "142a61061f64f7d67299d87b27ce7711dcdaf1641d6f6cd2fa784a0fe9ee94c2",
    "run_c

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Devuelve este ZIP y descarga el notebook con sus salidas.
